# Data Cleaning
### Dataset: Breast Cancer METABRIC

This notebook loads the raw METABRIC clinical dataset, audits its quality, fixes the issues found, and writes a clean file for downstream EDA and modeling.


In [13]:
# Imports
import numpy as np
import pandas as pd
from pathlib import Path

In [14]:
# Paths
PROJECT_ROOT = Path("..").resolve()
DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_PATH = DATA_DIR / "Breast_Cancer_METABRIC.csv"
CLEAN_DATA_PATH = DATA_DIR / "cleaned_data.csv"

print("Project Root:  ", PROJECT_ROOT)
print("Data Directory:", DATA_DIR)
print("Raw Data Path: ", RAW_DATA_PATH)
print("Clean Data Path:", CLEAN_DATA_PATH)

Project Root:   D:\study_material\ProjectFinal\CapstoneProject
Data Directory: D:\study_material\ProjectFinal\CapstoneProject\data
Raw Data Path:  D:\study_material\ProjectFinal\CapstoneProject\data\Breast_Cancer_METABRIC.csv
Clean Data Path: D:\study_material\ProjectFinal\CapstoneProject\data\cleaned_data.csv


In [15]:
# Load data
data = pd.read_csv(RAW_DATA_PATH)
df = data.copy()  # keep an untouched backup in `data`
df.head()

,Patient ID,Age at Diagnosis,Type of Breast Surgery,Cancer Type,Cancer Type Detailed,Cellularity,Chemotherapy,Pam50 + Claudin-low subtype,Cohort,ER status measured by IHC,...,Overall_Survival_Status,PR Status,Radio Therapy,Relapse Free Status (Months),Relapse Free Status,Sex,3-Gene classifier subtype,Tumor Size,Tumor Stage,Patient's_Vital_Status
0,MB-0000,75.65,Mastectomy,Breast Cancer,Breast Invasive Ductal Carcinoma,NaN,No,claudin-low,1.0,Positve,...,Living,Negative,Yes,138.65,Not Recurred,Female,ER-/HER2-,22.0,2.0,Living
1,MB-0002,43.19,Breast Conserving,Breast Cancer,Breast Invasive Ductal Carcinoma,High,No,LumA,1.0,Positve,...,Living,Positive,Yes,83.52,Not Recurred,Female,ER+/HER2- High Prolif,10.0,1.0,Living
2,MB-0005,48.87,Mastectomy,Breast Cancer,Breast Invasive Ductal Carcinoma,High,Yes,LumB,1.0,Positve,...,Deceased,Positive,No,151.28,Recurred,Female,NaN,15.0,2.0,Died of Disease
3,MB-0006,47.68,Mastectomy,Breast Cancer,Breast Mixed Ductal and Lobular Carcinoma,Moderate,Yes,LumB,1.0,Positve,...,Living,Positive,Yes,162.76,Not Recurred,Female,NaN,25.0,2.0,Living
4,MB-0008,76.97,Mastectomy,Breast Cancer,Breast Mixed Ductal and Lobular Carcinoma,High,Yes,LumB,1.0,Positve,...,Deceased,Positive,Yes,18.55,Recurred,Female,ER+/HER2- High Prolif,40.0,2.0,Died of Disease


## 1. Initial audit
Check shape, dtypes, duplicates, and missing values before touching anything.

In [16]:
print("Rows:", df.shape[0], " Columns:", df.shape[1])
print("\nDuplicate rows:", df.duplicated().sum())
print("\nData types:\n", df.dtypes.value_counts())

Rows: 2509  Columns: 34

Duplicate rows: 0

Data types:
 object     24
float64    10
Name: count, dtype: int64


In [17]:
null_summary = pd.DataFrame({
    "nulls": df.isnull().sum(),
    "missing_percentage": (df.isnull().mean() * 100).round(2)
}).sort_values("missing_percentage", ascending=False)

null_summary[null_summary["nulls"] > 0]

,nulls,missing_percentage
3-Gene classifier subtype,745,29.69
Tumor Stage,721,28.74
Primary Tumor Laterality,639,25.47
Cellularity,592,23.60
Type of Breast Surgery,554,22.08
Inferred Menopausal State,529,21.08
Integrative Cluster,529,21.08
Hormone Therapy,529,21.08
Pam50 + Claudin-low subtype,529,21.08
Chemotherapy,529,21.08


No duplicate rows. Missing values range from <1% up to ~30% of rows, concentrated in a handful of clinical/genomic fields. These will be handled during EDA/preprocessing with a proper imputer (fit on training data only) rather than here, since blanket-imputing the whole file at the cleaning stage would leak information across an eventual train/test split.

## 2. Restrict to the intended population
`Cancer Type` should be uniformly `Breast Cancer` for this dataset, but 3 rows are labeled `Breast Sarcoma` — a different disease that doesn't belong in a breast-carcinoma study. These rows are dropped rather than silently kept as noise.

In [18]:
print(df["Cancer Type"].value_counts(dropna=False))

df = df[df["Cancer Type"] == "Breast Cancer"].copy()
print("\nRows after filtering:", df.shape[0])

Cancer Type
Breast Cancer     2506
Breast Sarcoma       3
Name: count, dtype: int64

Rows after filtering: 2506


## 3. Drop the target-less rows
`Overall_Survival_Status` is the intended prediction target. Rows without a label can't be used for supervised training or for label-conditioned EDA, so they're removed.

In [19]:
print("Missing target rows:", df["Overall_Survival_Status"].isnull().sum())

df = df.dropna(subset=["Overall_Survival_Status"]).copy()
print("Rows after dropping missing-target rows:", df.shape[0])

Missing target rows: 528
Rows after dropping missing-target rows: 1978


## 4. Remove irrelevant / redundant / leaky columns
- **`Patient ID`** — a unique identifier, not a feature. Kept as the index for traceability instead of being dropped outright, so records stay identifiable without being fed to a model.
- **`Sex`** — constant (100% `Female`); zero predictive variance.
- **`Cancer Type`** — now constant after the filtering step above.
- **`ER status measured by IHC`** — duplicates `ER Status` and contains a typo category (`Positve`); the clean `ER Status` column is kept instead.
- **`HER2 status measured by SNP6`** — a secondary/experimental measurement of the same clinical fact captured more reliably in `HER2 Status`.
- **`Patient's_Vital_Status`** — a 3-category variant of the target itself (`Living` / `Died of Disease` / `Died of Other Causes`); keeping it alongside `Overall_Survival_Status` would leak the label.
- **`Cohort`** — a study-accrual batch ID, not a clinical or biological measurement; keeping it risks the model learning batch effects instead of signal.

In [20]:
df = df.set_index("Patient ID")

cols_to_drop = [
    "Sex",
    "Cancer Type",
    "ER status measured by IHC",
    "HER2 status measured by SNP6",
    "Patient's_Vital_Status",
    "Cohort", "Overall Survival (Months)", 
    "Relapse Free Status", 
    "Relapse Free Status (Months)"
]
df = df.drop(columns=cols_to_drop)

print("Remaining columns (%d):\n" % df.shape[1])
print(df.columns.to_list())

Remaining columns (24):

['Age at Diagnosis', 'Type of Breast Surgery', 'Cancer Type Detailed', 'Cellularity', 'Chemotherapy', 'Pam50 + Claudin-low subtype', 'ER Status', 'Neoplasm Histologic Grade', 'HER2 Status', 'Tumor Other Histologic Subtype', 'Hormone Therapy', 'Inferred Menopausal State', 'Integrative Cluster', 'Primary Tumor Laterality', 'Lymph nodes examined positive', 'Mutation Count', 'Nottingham prognostic index', 'Oncotree Code', 'Overall_Survival_Status', 'PR Status', 'Radio Therapy', '3-Gene classifier subtype', 'Tumor Size', 'Tumor Stage']


## 6. Tidy dtypes
`Neoplasm Histologic Grade` and `Tumor Stage` are ordinal clinical scores, not continuous measurements — cast them to a nullable integer type so they display and behave as discrete grades (NaNs are preserved for later imputation).

In [21]:
ordinal_cols = ["Neoplasm Histologic Grade", "Tumor Stage"]
for c in ordinal_cols:
    df[c] = df[c].astype("Int64")

df[ordinal_cols].describe()

,Neoplasm Histologic Grade,Tumor Stage
count,1893.0,1464.0
mean,2.414157,1.737705
std,0.649158,0.641085
min,1.0,0.0
25%,2.0,1.0
50%,3.0,2.0
75%,3.0,2.0
max,3.0,4.0


## 7. Final check

In [22]:
print("Final shape:", df.shape)
print("\nDuplicate rows:", df.duplicated().sum())
print("\nRemaining missing values:\n")
print(df.isnull().sum()[df.isnull().sum() > 0].sort_values(ascending=False))
df.head()

Final shape: (1978, 24)

Duplicate rows: 0

Remaining missing values:

Tumor Stage                       514
3-Gene classifier subtype         216
Mutation Count                    119
Primary Tumor Laterality          110
Neoplasm Histologic Grade          85
Lymph nodes examined positive      74
Cellularity                        63
Tumor Other Histologic Subtype     42
Type of Breast Surgery             25
Tumor Size                         23
Chemotherapy                        1
Integrative Cluster                 1
Inferred Menopausal State           1
Hormone Therapy                     1
HER2 Status                         1
Pam50 + Claudin-low subtype         1
Nottingham prognostic index         1
PR Status                           1
Radio Therapy                       1
dtype: int64


,Age at Diagnosis,Type of Breast Surgery,Cancer Type Detailed,Cellularity,Chemotherapy,Pam50 + Claudin-low subtype,ER Status,Neoplasm Histologic Grade,HER2 Status,Tumor Other Histologic Subtype,...,Lymph nodes examined positive,Mutation Count,Nottingham prognostic index,Oncotree Code,Overall_Survival_Status,PR Status,Radio Therapy,3-Gene classifier subtype,Tumor Size,Tumor Stage
Patient ID,,,,,,,,,,,,,,,,,,,,,
MB-0000,75.65,Mastectomy,Breast Invasive Ductal Carcinoma,NaN,No,claudin-low,Positive,3,Negative,Ductal/NST,...,10.0,NaN,6.044,IDC,Living,Negative,Yes,ER-/HER2-,22.0,2
MB-0002,43.19,Breast Conserving,Breast Invasive Ductal Carcinoma,High,No,LumA,Positive,3,Negative,Ductal/NST,...,0.0,2.0,4.020,IDC,Living,Positive,Yes,ER+/HER2- High Prolif,10.0,1
MB-0005,48.87,Mastectomy,Breast Invasive Ductal Carcinoma,High,Yes,LumB,Positive,2,Negative,Ductal/NST,...,1.0,2.0,4.030,IDC,Deceased,Positive,No,NaN,15.0,2
MB-0006,47.68,Mastectomy,Breast Mixed Ductal and Lobular Carcinoma,Moderate,Yes,LumB,Positive,2,Negative,Mixed,...,3.0,1.0,4.050,MDLC,Living,Positive,Yes,NaN,25.0,2
MB-0008,76.97,Mastectomy,Breast Mixed Ductal and Lobular Carcinoma,High,Yes,LumB,Positive,3,Negative,Mixed,...,8.0,2.0,6.080,MDLC,Deceased,Positive,Yes,ER+/HER2- High Prolif,40.0,2


In [23]:
# Save cleaned data
df.to_csv(CLEAN_DATA_PATH)
print("Saved to:", CLEAN_DATA_PATH)

Saved to: D:\study_material\ProjectFinal\CapstoneProject\data\cleaned_data.csv
